# 01 · Entrenamiento en Colab (opcional)

Alternativa a una GPU local: ejecuta en Colab las corridas del experimento y las semillas elegidos. Los resultados se escriben en `DRIVE_DIR/results` (Google Drive) y la ejecución es **reanudable**: si Colab se desconecta, vuelve a ejecutar todo el notebook y las corridas terminadas se omiten.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **GPU** (por ejemplo, T4).

Para unir después estos resultados con los de otra máquina: descarga `DRIVE_DIR/results` y ejecuta `python -m src.merge_runs <carpeta o .zip>`.

In [ ]:
EXPERIMENTS = "E1"  # E1, E2, E3 o varios separados por coma: "E1,E3"
SEEDS = "0-4"       # rango "0-3", lista "0,2,4" o una sola semilla "4"

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Rosa-SDev/roya-cafe-cbam-vs-deit.git"
DRIVE_DIR = Path("/content/drive/MyDrive/IA_ProyectoFinal_RoCoLe")  # carpeta de Drive para datos y resultados (solo Colab)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = Path("/content/codigo")
    if not REPO.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Repositorio:", REPO, "| commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
      capture_output=True, text=True).stdout.strip())

In [ ]:
import torch
assert torch.cuda.is_available(), "No hay GPU: cambia el tipo de entorno a GPU"
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
# Datos: copia la caché desde Drive (la genera el notebook 00) o la crea aquí
DATA_ROOT = Path("/content/data")
RESULTS_ROOT = DRIVE_DIR / "results"
if not (DATA_ROOT / "rocole_256.npz").exists():
    cached = DRIVE_DIR / "data"
    if (cached / "rocole_256.npz").exists():
        DATA_ROOT.mkdir(parents=True, exist_ok=True)
        subprocess.run(["cp", str(cached / "rocole_256.npz"), str(cached / "metadata.csv"), str(DATA_ROOT)], check=True)
    else:  # plan B: descargar de Mendeley y generar la caché aquí
        subprocess.run([sys.executable, "-m", "src.prepare_data", "--data-root", str(DATA_ROOT), "--download"], check=True)
os.environ["ROCOLE_DATA_ROOT"] = str(DATA_ROOT)
os.environ["ROCOLE_RESULTS_ROOT"] = str(RESULTS_ROOT)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print("Datos:", DATA_ROOT, "| resultados:", RESULTS_ROOT)

## (Opcional, solo la primera vez) Benchmark de tiempos

Entrena las 6 configuraciones 3 épocas (unos 5 min) y estima minutos por corrida y horas por experimento en esta GPU. Los resultados del benchmark van a una carpeta local, no a Drive.

In [ ]:
RUN_BENCHMARK = False
if RUN_BENCHMARK:
    !python -m src.runs --benchmark --results-root /content/bench

## Corridas seleccionadas

In [ ]:
!python -m src.runs --experiment {EXPERIMENTS} --seeds {SEEDS} --results-root "{RESULTS_ROOT}"

## Ejecutar

Ejecuta las corridas pendientes de la selección, en orden. Cada corrida escribe en `DRIVE_DIR/results` su JSON (`runs/<experimento>/`), los mapas de ejemplo (`*_maps.npz`) y el mejor modelo (`checkpoints/`, que no se sube a git).

In [ ]:
!python -m src.runs --experiment {EXPERIMENTS} --seeds {SEEDS} --execute

## Avance de todas las corridas

In [ ]:
!python -m src.runs --status --results-root "{RESULTS_ROOT}"